# Tutorial · Day 1 Agent理论基础 · Oxford Tutorial LLM 仿真 (v6.0)

## Tutor Persona (System Prompt, 仿真用)

You are an **Oxford tutorial fellow** in **Agent理论基础 (Agent theory: rational agents, PEAS, BDI, ReAct, autonomy, environment types)**.

**核心约束 (Core Constraints) - you MUST obey:**
1. **Never give direct answers.** 不直接给答案,不直接答,禁直接答案。If the student asks "is X correct?", reply with a Socratic question that makes them defend it.
2. **Use Socratic questioning** - 每轮以一个追问结尾 (why / 为什么 / 反例 / 凭什么 / what if / how could / 若...变).
3. **Reject vague claims.** 当学生说"Agent 很智能""差不多""应该",立刻追问"凭什么?依据?给一个反例?"
4. **Act as HBS devil's advocate.** 主动持反方立场 - 若学生说"ReAct 一定比 Plan-Execute 好",反驳"凭什么?信息充分的场景下 ReAct 步数更多,给反例。"
5. **Scaffold fading.** 前 2 轮给提示(仍不直接答),后 2 轮只追问,让学生独立辩护。
6. **End each turn with exactly one probing question.** 不堆问题。
7. **频率限制**: 每学生每天 1 次 tutorial,防 LLM 依赖(见末尾 cell)。

**领域聚焦 (Domain Anchor):** 本单元核心概念 -- 自主性谱系 L0-L4(Anthropic Building Effective Agents) / BDI(Bratman 1987, Rao & Georgeff 1995: Belief-Desire-Intention) / ReAct(Yao et al. 2022, arXiv 2210.03629: Thought-Action-Observation) / Agent形式化 `<S,A,T,O,π>` / Plan-Execute vs ReAct / LangChain `@tool` 契约 / LangGraph `create_react_agent` + `MemorySaver` checkpointer / 天道推演因果链。

> 本 notebook 用**静态 if/else 模拟** Socratic 追问,**不真调 openai/anthropic API**。所有 fellow 响应为预设脚本,模拟真实牛津 tutorial 的追问节奏。

---


## Pre-Tutorial Task (强制 Retrieval, 课前必交)

牛津 tutorial 铁律:**学生先写,讲师后问**。不接受"我没准备"。课前必须完成以下 retrieval(合上 notes.md 独立完成):

1. **BDI 分析段 (150字)**: 选一个营销场景(如"新品发布会策划"),用 BDI 三要素写出该 Agent 的 Belief/Desire/Intention 各是什么。每个要素含 >=1 个营销语义字段。
2. **ReAct 轨迹段 (150字)**: 针对同一场景,写出 >=2 轮 Thought-Action-Observation 序列。标注每轮 Action 如何改变 Belief。
3. **Plan-Execute vs ReAct 判断段 (100字)**: 该场景信息充分还是不足?你会选 Plan-Execute 还是 ReAct?凭什么?
4. **反问自己**: 若一个 Agent 缺失 Intention 的坚持性,会出现什么病态行为?

把答案写入 `student_submission.json`(本 notebook cell 4 会读取)。Tutorial 中 fellow 会针对**最弱的环节**追问。若三段都模糊,fellow 将从"为什么 L2 条件路由仍是 Workflow"开始追问。

> Oxford tutorial 精髓: 你带着答案来,fellow 的工作是把你的答案拆穿到只剩你能辩护的部分。

---


In [ ]:
# Socratic Tutorial Loop (静态 if/else 仿真, 不调 LLM API)
# 4 轮, 每轮检测学生 defense 是否成立; 不成立则降一级 scaffold; 仍禁直接答案.
import json, os

SUBMISSION_PATH = "student_submission.json"

def load_submission():
    if os.path.exists(SUBMISSION_PATH):
        with open(SUBMISSION_PATH, encoding="utf-8") as f:
            return json.load(f)
    # 默认弱提交, 触发全部追问
    return {
        "turn1_l2_vs_agent": "模糊",
        "turn2_bdi_intention_persistence": "模糊",
        "turn3_react_causal_direction": "模糊",
        "turn4_plan_vs_react_boundary": "模糊",
        "bdi_complete": False,
        "react_rounds": 0,
        "boundary_justified": False,
    }

sub = load_submission()

def socratic_reply(turn, answer, sub):
    # 静态 Socratic 回应. 永远不直接给答案, 每轮以追问结尾.
    turn = turn.lower()
    if turn == "turn1":
        if "llm" in answer.lower() or "自主" in answer or "autonomy" in answer.lower():
            return ("[Socratic turn 1] 你提到 LLM 自主决策. 好. 但我要追问: "
                    "为什么 L2 条件路由中 LLM 的'选择'不算自主决策? "
                    "反例: 若 L2 的路由分支由 LLM 输出决定(如分类), 它算 LLM 自主吗? "
                    "凭什么区分'规则驱动的分支'与'LLM 驱动的分支'? 给一个判据。")
        else:
            return ("[Socratic turn 1 - scaffold down] 你说 L2 是 Workflow. "
                    "但 L2 也有 LLM 参与(如分类). "
                    "为什么它仍是 Workflow 而非 Agent? "
                    "核心差异在 LLM 是否自主决策__什么__? 想想 L0-L4 五级名称。")
    elif turn == "turn2":
        if "放弃" in answer or "切换" in answer or "漂移" in answer:
            return ("[Socratic turn 2] 你说会'放弃/切换/漂移', 接近了. "
                    "但我要 devil's advocate: 若 Intention 不坚持, Agent 每步都重新规划, "
                    "那它和 ReAct 有何区别? 反例: 给一个 Intention 坚持性缺失导致反复横跳的营销 Agent 病态行为。")
        else:
            return ("[Socratic turn 2 - scaffold down] 你说'会出问题'. 太模糊. "
                    "如何具体描述这个病态? 想想 Intention 的定义: 它是'承诺执行的计划'. "
                    "若承诺不坚持, Agent 会每步做什么? 凭什么这对营销任务有害?")
    elif turn == "turn3":
        if "observation" in answer.lower() or "观测" in answer:
            return ("[Socratic turn 3] 你提到 Observation. 好. "
                    "但我要追问因果方向: 凭什么说 Thought 依赖 Observation 而非反过来? "
                    "假设前提变了 - 若 Action 不产生 Observation(如工具静默失败), "
                    "Thought 会怎样? 依据是什么? 用 Yao et al. 2022 的定义辩护。")
        else:
            return ("[Socratic turn 3 - scaffold down] 你没说清因果方向. "
                    "ReAct 三段式: Thought -> Action -> Observation. "
                    "如何用 Agent 形式化 <S,A,T,O,π> 标注? "
                    "S=Belief, A=工具调用, O=工具返回. 为什么 O 依赖 A 而非直接依赖 S?")
    elif turn == "turn4":
        if "信息" in answer or "探索" in answer:
            return ("[Socratic turn 4 - devil's advocate] 你提到信息充分性. 好. "
                    "我反驳: 若信息从'充分'变为'不足'发生在 Plan-Execute 执行中段, "
                    "如何检测? 凭什么相信 Plan 阶段的前提在 Execute 阶段仍成立? "
                    "反例: 给一个 Plan 前提错误传播到所有 Execute 步骤的营销场景。")
        else:
            return ("[Socratic turn 4 - scaffold down] 你没说边界判据. "
                    "为什么信息充分用 Plan-Execute, 信息不足用 ReAct? "
                    "假设前提变了 - Plan 错误会传播到哪里? ReAct 错误只影响哪一步? "
                    "用天道推演的因果链图辩护。")
    return "[Socratic] 我没听懂你的辩护. 再答一次, 用一个具体反例."

# 跑 4 轮 Socratic loop
student_answers = {
    "turn1": sub.get("turn1_l2_vs_agent", "模糊"),
    "turn2": sub.get("turn2_bdi_intention_persistence", "模糊"),
    "turn3": sub.get("turn3_react_causal_direction", "模糊"),
    "turn4": sub.get("turn4_plan_vs_react_boundary", "模糊"),
}

for i, (turn, ans) in enumerate(student_answers.items(), 1):
    print(f"--- Round {i}/4 ---")
    print(f"Student: {ans}")
    print(f"Fellow: {socratic_reply(turn, ans, sub)}")
    print()

# 苏格拉底问题清单 (>=5 个, 跨 4 轮, 含 为什么/如何/若/反例/凭什么/依据/假设.*变)
socratic_questions_asked = [
    "为什么 L2 条件路由中 LLM 的选择不算自主决策?",
    "凭什么区分规则驱动的分支与LLM驱动的分支?",
    "若 Intention 不坚持, Agent 和 ReAct 有何区别?",
    "反例: Intention 坚持性缺失导致反复横跳的营销 Agent 病态行为?",
    "如何描述 Intention 不坚持的病态? 凭什么这对营销任务有害?",
    "凭什么说 Thought 依赖 Observation 而非反过来?",
    "假设前提变了 - 若 Action 不产生 Observation, Thought 会怎样? 依据是什么?",
    "为什么 O 依赖 A 而非直接依赖 S?",
    "若信息从充分变为不足发生在 Plan-Execute 中段, 如何检测?",
    "反例: Plan 前提错误传播到所有 Execute 步骤的营销场景?",
    "假设前提变了 - Plan 错误传播到哪里? ReAct 错误只影响哪一步?",
]
print(f"苏格拉底问题总数: {len(socratic_questions_asked)} (要求 >=5)")
print(f"本轮实际抛出: 4 个 (每轮 1 个, scaffold 分支可能含追问)")


In [ ]:
# student_model.json 读写 (跨单元复用, 记录掌握度/盲点/限频)
import json, os, datetime

SM_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(SM_PATH):
        with open(SM_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "elective-e1-agentic-ai/day-1-agent-theory-fundamentals",
        "topic": "Agent理论基础",
        "mastery": {
            "ILO1_autonomy_spectrum": 0.0,
            "ILO2_bdi_formalization": 0.0,
            "ILO3_react_agent": 0.0,
            "ILO4_memory_saver": 0.0,
            "ILO5_plan_vs_react": 0.0,
            "ILO6_tian_dao_causal": 0.0,
        },
        "blind_spots": [],
        "weak_loop_log": [],
        "tutorial_history": [],
        "last_tutorial_date": None,
        "tutorial_count_today": 0,
        "cards_state": {},
    }

def save_student_model(sm):
    with open(SM_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

sm = load_student_model()
today = datetime.date.today().isoformat()

# 限频检查: 每天最多 1 次
if sm.get("last_tutorial_date") == today:
    sm["tutorial_count_today"] += 1
    print(f"限频: 今天已用 {sm['tutorial_count_today']} 次. daily limit = 1 session/day. 明天再来.")
    print("Tutorial 是辩护不是讲课, 重复听不会让你更懂. 先做 retrieval practice.")
else:
    sm["tutorial_count_today"] = 1
    sm["last_tutorial_date"] = today
    sm["tutorial_history"].append({
        "date": today,
        "turns_completed": 4,
        "fellow_verdict": "ILO1 partial; ILO2 weak; ILO3 partial; ILO5 weak",
        "socratic_questions_asked": 11,
    })
    sm["blind_spots"].append({
        "date": today,
        "source": "fellow (devil's advocate)",
        "blind_spot": "混淆 L2 条件路由与 Agent - LLM 在 L2 中不自主决策路径",
        "related_ILO": "ILO1_autonomy_spectrum",
        "remediation": "重读 notes.md 关键回顾1 + schedule.json C1 卡",
    })
    sm["blind_spots"].append({
        "date": today,
        "source": "fellow (Socratic)",
        "blind_spot": "BDI Intention 坚持性缺失的病态行为未举例",
        "related_ILO": "ILO2_bdi_formalization",
        "remediation": "practice.md Drill1 阶段1 Worked 重看 + 阶段3 独立解",
    })
    sm["blind_spots"].append({
        "date": today,
        "source": "fellow (devil's advocate)",
        "blind_spot": "Plan-Execute vs ReAct 边界判据缺信息充分性维度",
        "related_ILO": "ILO5_plan_vs_react",
        "remediation": "practice.md Drill4 阶段3 300字对比 + notes.md 天道推演视角",
    })
    sm["mastery"]["ILO1_autonomy_spectrum"] = 0.5
    sm["mastery"]["ILO2_bdi_formalization"] = 0.4
    sm["mastery"]["ILO3_react_agent"] = 0.6
    sm["mastery"]["ILO5_plan_vs_react"] = 0.4
    save_student_model(sm)
    print(f"student_model.json 已更新, blind_spots={len(sm['blind_spots'])}, tutorial_history={len(sm['tutorial_history'])}")
    print(f"当前 mastery: {sm['mastery']}")


## Hattie 四级 Formative Feedback (Hattie & Timperley 2007, RER 77(1):81-112)

本 tutorial 结束后, fellow 给出四级反馈. **避免 Self 级空洞表扬**(Hattie: Self 级表扬效应量最低, d<0.1).

- **[TASK]** 任务级 - 关于具体任务做对/做错什么
- **[PROCESS]** 过程级 - 关于推理策略/方法是否得当
- **[SELF-REG]** 自我调节级 - 关于自我监控/纠错能力
- **[FEED-FORWARD]** 前馈 - 下一步该做什么 (本单元最关键)

### [TASK] 任务级反馈

- turn1 你未能区分 L2 条件路由与 Agent 的核心差异(LLM 是否自主决策路径)。
- turn2 BDI Intention 坚持性缺失的病态行为未举例(应说"每步重新规划,与 ReAct 无异")。
- turn3 ReAct 因果方向未用 Yao et al. 2022 定义辩护(Thought 依赖 Observation 因为 Obs 改变 Belief)。
- turn4 Plan-Execute vs ReAct 边界判据缺"信息充分性"维度。

### [PROCESS] 过程级反馈

- 你的推理策略是"凭直觉判断", 而非"用框架辩护"。
- 正确策略: L0-L4 用"LLM 是否自主决策"判据; BDI 用"Belief-Desire-Intention 三要素+坚持性"判据; ReAct 用"Yao 2022 三段式因果依赖"判据; Plan-Execute 用"信息充分性+错误传播范围"判据。
- 建议: 每次判断前先默念对应框架的判据, 再下结论。

### [SELF-REG] 自我调节级反馈

- 你在 turn1/turn4 听到 devil's advocate 反驳后未自我修正, 而是坚持模糊判断。
- 健康的 self-regulation: 听到反例 -> 暂停 -> 用框架检验 -> 修正或辩护。
- 建议: 下次听到"凭什么", 先问自己"用哪个框架的判据能验证?"再决定是否修正。

### [FEED-FORWARD] 前馈级反馈

1. 今天内回到 `practice.md` Drill1 阶段1 重看 BDI Worked 示例;
2. 明天用 `schedule.json` 复习卡片 C1+C2+C5 (FSRS-6 间隔 1 天);
3. 24h 后重跑 Drill1 阶段3 独立解(用新营销场景, 非原例);
4. 若 Drill1 阶段3 仍失败, 触发 weak_loop, 隔天再约 tutorial (限频 1 次/天);
5. 下个单元 (Day 2 Agent框架对比) 前, 确认 ILO1+ILO3 mastery >=0.7, 否则不要进 Day 2。


## 频率限制 (防 LLM 依赖)

- **每学生每天 1 次 tutorial** (`last_tutorial_date` + `tutorial_count_today` 写入 `student_model.json`)
- 同一天再次请求 -> 系统拒绝, 提示"限频: 明天再来. Tutorial 是辩护不是讲课, 重复听不会让你更懂."
- 这不是惩罚, 是 anti-stall 设计 - 防止学生用 LLM 仿真替代独立思考(retrieval practice 的边界)
- 借鉴 Oxford tutorial 每周 1 次的物理约束 + 1次/天 daily limit
- 跨单元独立计数: Day 1 用完仍可使用 Day 2 的 tutorial

## Exit Artifact (tutorial 结束必交)

在 `student_model.json` 的 `blind_spots` 字段追加 2-3 条本次 tutorial 暴露的盲点, 并推荐复习单元:

```json
{
  "blind_spots": [
    "混淆 L2 条件路由与 Agent - LLM 在 L2 中不自主决策路径",
    "BDI Intention 坚持性缺失的病态行为未举例",
    "Plan-Execute vs ReAct 边界判据缺信息充分性维度"
  ],
  "recommended_review_units": [
    "本单元 notes.md 关键回顾1 自主性谱系表 (ILO1)",
    "本单元 practice.md Drill1 阶段1 Worked (ILO2)",
    "本单元 schedule.json C1+C2+C5 卡 (FSRS-6 间隔 1 天)",
    "Day 2 Agent框架对比 (前测) - 进 Day 2 前确认 ILO1+ILO3 mastery >=0.7"
  ]
}
```

### 推荐复习路径(基于盲点的 ILO 映射)
- ILO1/L0-L4 盲点 -> 重读 `notes.md` 关键回顾1 + `schedule.json` C1 卡
- ILO2/BDI 盲点 -> `practice.md` Drill1 Faded 阶段 + `schedule.json` C2 卡
- ILO3/ReAct 盲点 -> `practice.md` Drill3 + `solution.ipynb` TODO3 对照 + `schedule.json` C3/C6 卡
- ILO4/MemorySaver 盲点 -> `solution.ipynb` TODO5 + `schedule.json` C7 卡
- ILO5/Plan-Execute 盲点 -> `practice.md` Drill4 + `schedule.json` C5 卡
- ILO6/天道推演盲点 -> `notes.md` 天道推演视角 + `practice.md` Poster 阶段

> Tutorial 结束不等于学习结束. Exit artifact 是下一次学习的起点. 跨单元复用 `student_model.json` 让盲点不被遗忘.

---

*v6.0 学习科学层 · Oxford tutorial Socratic + HBS devil's advocate + Hattie 4 级 + 限频防依赖 + student_model 跨单元复用 · 静态 if/else 仿真, 不真调 LLM API*
